# Perform variant enrichment analysis based on local structural features

In [10]:
import polars as pl
import pandas as pd
from statsmodels.stats.multitest import multipletests
from scipy.stats import hypergeom
from tqdm import tqdm

In [2]:
inputs_dir = "../1_inputs"
outputs_dir = "../3_outputs"
metadata_outputs = "../../1_allele_collection/3_outputs"

In [5]:
scores = pl.read_csv(f"{outputs_dir}/assay_scores.csv")
annotations = pl.read_csv(f"{metadata_outputs}/sequence_feature_annotations/3_allele_annotations.csv")

In [14]:
# Define function for performing enrichment analysis
def overrepresentation_analysis(target_set_library: pl.DataFrame, target_hits: list, universe: list):
    target_hits_set = set(target_hits)
    universe_set = set(universe)
    sets = target_set_library.select(pl.col("feat_type")).unique().to_series()
    results = []

    # For each target set, calculate the overlap and perform the hypergeometric test
    for target_set in tqdm(sets):
        targets_in_set = set(target_set_library.filter(pl.col("feat_type") == target_set)["allele_0"].to_list())
        
        overlap = target_hits_set.intersection(targets_in_set)
        overlap_size = len(overlap)

        M = len(universe_set)  # Total number of genes in the universe
        n = len(targets_in_set)  # Number of genes in the gene set
        N = len(target_hits_set)  # Number of gene hits
        x = overlap_size  # Number of overlapping genes (successes)

        p_value = hypergeom.sf(x - 1, M, n, N)

        results.append({
            "target_set": target_set,
            "overlap_size": overlap_size,
            "target_set_size": n,
            "hit_list_size": N,
            "universe_size": M,
            "p_value": p_value,
        })

    return pl.DataFrame(results)

# FDR adjust the p-values
def fdr_polars(df: pl.DataFrame):
    p_values = df.select("p_value").to_series().to_numpy()
    _, p_adjusted, _, _ = multipletests(p_values, method='fdr_bh', is_sorted=False)
    df = df.with_columns(pl.Series("fdr", p_adjusted))
    return df

In [ ]:
# Abundance
abun_scores = scores.filter(~pl.col("abun_hit").is_null())
abun_universe = abun_scores.select("allele_0").to_series().unique().to_list()
abun_hits = abun_scores.filter(pl.col("abun_hit")).select("allele_0").to_series().unique().to_list()

abun_enrich = overrepresentation_analysis(annotations, abun_hits, abun_universe)
abun_enrich = fdr_polars(abun_enrich).sort("p_value")

100%|██████████| 763/763 [00:00<00:00, 12477.74it/s]


In [17]:
# PPI
ppi_scores = scores.filter(~pl.col("ppi_hit").is_null())
ppi_universe = ppi_scores.select("allele_0").to_series().unique().to_list()
ppi_hits = ppi_scores.filter(pl.col("ppi_hit")).select("allele_0").to_series().unique().to_list()

ppi_enrich = overrepresentation_analysis(annotations, ppi_hits, ppi_universe)
ppi_enrich = fdr_polars(ppi_enrich).sort("p_value")

100%|██████████| 763/763 [00:00<00:00, 11199.65it/s]


In [18]:
# misloc
misloc_scores = scores.filter(~pl.col("misloc_hit").is_null())
misloc_universe = misloc_scores.select("allele_0").to_series().unique().to_list()
misloc_hits = misloc_scores.filter(pl.col("misloc_hit")).select("allele_0").to_series().unique().to_list()

misloc_enrich = overrepresentation_analysis(annotations, misloc_hits, misloc_universe)
misloc_enrich = fdr_polars(misloc_enrich).sort("p_value")

100%|██████████| 763/763 [00:00<00:00, 11811.02it/s]
